# Phase 10 — Tuned Model Interpretation

## TL;DR

- Crop type is the tuned forest's dominant predictive input: shuffling `item` increases outer-validation MAE by about 60.1k hg/ha.
- Temperature, rainfall, pesticide totals, and country are also important to predictions, but this does not mean they cause the observed yield differences.
- Rainfall is constant through time within each country, so rainfall importance is entangled with location. Pesticides are national totals rather than crop- or farm-level application rates.
- Shuffling validation years causes no prediction change. All validation years are beyond the training range, so they follow the same tree branches.
- Holding other inputs fixed, example predictions are identical for 2008, 2010, 2020, and 2030. The forest therefore cannot extrapolate a long-term time trend.
- The model is a strong historical interpolation and conditional-prediction candidate, but not yet a defensible standalone long-horizon forecasting model.
- Test data remains untouched.


## Context & Methods

This notebook asks three questions:

1. Which original input columns does the fitted model rely on most?
2. Do two interpretation methods tell a broadly consistent story?
3. Can the model respond meaningfully to years beyond its training period?

Grouped permutation importance is the primary method. Each original validation column is shuffled ten times while every other column is held in place. The resulting increase in MAE measures how much predictive information the fitted model loses. Shuffling happens before preprocessing, so all one-hot columns belonging to `area` or `item` move together.

Native random-forest importance is used only as a secondary consistency check because impurity-based importance can favour continuous variables or high-cardinality categories.

### Chart contract

- Question: which six original features most affect validation error when disrupted?
- Form: sorted horizontal bar chart with uncertainty whiskers.
- Unit: mean increase in validation MAE, hg/ha, across ten permutations.
- Palette: one dark-blue root with charcoal error bars; no redundant legend.
- Scale: zero-based because the bars compare absolute non-negative magnitudes.

### Key assumptions and limitations

- Importance describes the fitted model, not biological or economic causation.
- Correlated or proxy features can share, hide, or duplicate importance.
- Shuffling rainfall independently creates some country–rainfall combinations absent from the source data; rainfall importance must therefore be read cautiously.
- Scenario sensitivity changes one feature while holding others fixed. It diagnoses model behaviour and is not a policy or agronomic recommendation.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from crop_yield.evaluation import regression_metrics
from crop_yield.interpretation import (
    grouped_native_feature_importance,
    grouped_permutation_importance,
    prediction_sensitivity,
)
from crop_yield.models import build_tuned_random_forest_pipeline
from crop_yield.preprocessing import split_features_target
from crop_yield.splitting import temporal_train_validation_test_split

plt.style.use("seaborn-v0_8-whitegrid")

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_PATH = REPO_ROOT / "data/processed/crop_yield_modeling.csv"


## Data

### 1. Fit the tuned forest without accessing test outcomes


In [ ]:
crop_yield = pd.read_csv(DATA_PATH)
temporal_split = temporal_train_validation_test_split(crop_yield)
X_train, y_train = split_features_target(temporal_split.train)
X_validation, y_validation = split_features_target(
    temporal_split.validation
)

tuned_model = build_tuned_random_forest_pipeline()
tuned_model.fit(X_train, y_train)
validation_predictions = tuned_model.predict(X_validation)
validation_metrics = regression_metrics(
    y_validation.to_numpy(),
    validation_predictions,
)

print(f"Training rows: {len(X_train):,}")
print(f"Interpretation rows: {len(X_validation):,}")
print(f"Reserved test rows: {len(temporal_split.test):,}")
print(
    "Outer-validation metrics: "
    f"MAE={validation_metrics['mae']:,.2f}, "
    f"RMSE={validation_metrics['rmse']:,.2f}, "
    f"R²={validation_metrics['r2']:.4f}"
)


## Results

### 2. Measure grouped permutation importance


In [ ]:
permutation_importance = grouped_permutation_importance(
    tuned_model,
    X_validation,
    y_validation,
    n_repeats=10,
    random_state=42,
)
print(permutation_importance.round(2).to_string(index=False))


In [ ]:
feature_labels = {
    "item": "Crop",
    "area": "Country",
    "year": "Year",
    "average_temperature_c": "Average temperature",
    "average_rainfall_mm_per_year": "Average rainfall",
    "pesticides_tonnes": "Pesticide total",
}
plot_importance = permutation_importance.sort_values(
    "mae_increase_mean",
    ascending=True,
).copy()
plot_importance["display_feature"] = plot_importance["feature"].map(
    feature_labels
)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(
    plot_importance["display_feature"],
    plot_importance["mae_increase_mean"],
    xerr=plot_importance["mae_increase_std"],
    color="#2F5D8A",
    edgecolor="#203040",
    error_kw={"ecolor": "#30343B", "capsize": 3},
)
ax.set_xlim(left=0)
ax.set_title("Grouped Permutation Importance on 2008–2010 Validation Data")
ax.set_xlabel("Mean increase in MAE after shuffling (hg/ha)")
ax.set_ylabel("Original model input")
plt.tight_layout()
plt.show()


`item` is clearly dominant. Shuffling crop identity forces the model to confuse crops with very different yield scales. Temperature, rainfall, pesticide totals, and country also carry substantial predictive information. These values must not be translated into claims such as “more pesticides cause higher yields”: the data are observational, pesticide values are national totals, and several predictors encode overlapping country and time information.


### 3. Reconcile with native tree importance


In [ ]:
native_importance = grouped_native_feature_importance(tuned_model)
print(native_importance.round(4).to_string(index=False))

native_plot = native_importance.sort_values(
    "native_importance",
    ascending=True,
).copy()
native_plot["display_feature"] = native_plot["feature"].map(
    feature_labels
)
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(
    native_plot["display_feature"],
    native_plot["native_importance"],
    color="#8AA6C1",
    edgecolor="#203040",
)
ax.set_xlim(left=0)
ax.set_title("Grouped Native Random-Forest Importance")
ax.set_xlabel("Share of total impurity reduction")
ax.set_ylabel("Original model input")
plt.tight_layout()
plt.show()


Both approaches rank crop identity first and assign meaningful weight to location and the three measured environmental or input variables. Native importance assigns some training importance to `year`, whereas permutation importance on 2008–2010 is exactly zero. The next diagnostic explains that apparent contradiction.


### 4. Diagnose the forest's year extrapolation


In [ ]:
scenario_years = [2008, 2010, 2015, 2020, 2030]
year_scenarios = []

for crop in ["Maize", "Cassava", "Potatoes"]:
    reference = X_validation.loc[
        X_validation["item"].eq(crop)
    ].iloc[[0]]
    crop_sensitivity = prediction_sensitivity(
        tuned_model,
        reference,
        feature="year",
        values=scenario_years,
    )
    crop_sensitivity.insert(0, "item", crop)
    year_scenarios.append(crop_sensitivity)

year_sensitivity = pd.concat(year_scenarios, ignore_index=True)
print(year_sensitivity.round(2).to_string(index=False))


Tree models divide the feature space at thresholds observed during training. Because training ends in 2007, every later year is above the learned year thresholds and follows the same branches when the other inputs are unchanged. Consequently, the forest does not extend a rising or falling time trend into 2020 or 2030. This is a structural limitation, not a software bug.


## Checks

### 5. Assert interpretation safeguards


In [ ]:
assert permutation_importance.iloc[0]["feature"] == "item"
assert permutation_importance.iloc[0]["mae_increase_mean"] > 50_000
assert np.isclose(
    permutation_importance.set_index("feature").loc[
        "year",
        "mae_increase_mean",
    ],
    0,
)
assert native_importance.iloc[0]["feature"] == "item"
assert np.isclose(native_importance["native_importance"].sum(), 1)
assert year_sensitivity.groupby("item")["prediction"].nunique().eq(1).all()

test_evaluated = False
assert not test_evaluated
print(
    "All Phase 10 interpretation checks passed. "
    f"The {len(temporal_split.test):,}-row test set remains untouched."
)


## Takeaways

1. Crop identity is indispensable to the current model because the ten crops occupy very different yield distributions.
2. Country, rainfall, temperature, and pesticide totals improve historical prediction, but their importance mixes genuine signal with location, scale, technology, and time proxies.
3. Rainfall cannot be interpreted as a within-country annual weather effect because it is constant through time for each country in this dataset.
4. Pesticide importance cannot be interpreted as a crop-response effect because the feature is a country-level total and is not normalized by cropped area.
5. The random forest does not extrapolate calendar-year trends beyond 2007. The app must either restrict and explain its prediction scope or the project must add a forecasting component designed for extrapolation.
6. Because this affects the intended product, the final test set should remain sealed until the deployment scope or forecasting redesign is decided.
